# Chicago Food Inspection Analysis

## Data Exploration

### Environment Setup

In [1]:
# Standard library
from pathlib import Path

# Third-party
import pandas as pd
import numpy as np

# Project 
from food_inspection.config import config
from food_inspection.logging_config import get_logger

# Notebook display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

logger = get_logger("notebooks.01_data_exploration")
logger.info("Notebook 01_data_exploration started")

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Project root: {config.paths.project_root}")

2026-09-16 01:04:59,341 | food_inspection.config | INFO | Configuration loaded from: ..\config.yaml
2026-09-16 01:04:59,343 | notebooks.01_data_exploration | INFO | Notebook 01_data_exploration started
Pandas: 3.0.5
NumPy: 2.5.3
Project root: D:\ML\Portfolio\Projects\chicago-food-inspection-analysis


### Load Dataset

In [ ]:
raw_csv_path = config.paths.raw_data_dir / config.data["raw_filename"]
logger.info(f"Loading: {raw_csv_path.relative_to(config.paths.project_root)}")

# Load with everything as strings — preserve the raw mess
df = pd.read_csv(raw_csv_path, low_memory=False)

logger.info(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

2026-09-16 01:10:04,353 | notebooks.01_data_exploration | INFO | Loading: data\raw\Food_Inspections_20260914.csv
2026-09-16 01:10:09,133 | notebooks.01_data_exploration | INFO | Loaded 314,555 rows × 17 columns
Shape: (314555, 17)
Rows: 314,555
Columns: 17


### Basic Info

In [10]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 314555 entries, 0 to 314554
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Inspection ID    314555 non-null  str    
 1   DBA Name         314550 non-null  str    
 2   AKA Name         312153 non-null  str    
 3   License #        314531 non-null  float64
 4   Facility Type    309259 non-null  str    
 5   Risk             314463 non-null  str    
 6   Address          314550 non-null  str    
 7   City             314372 non-null  str    
 8   State            314475 non-null  str    
 9   Zip              314509 non-null  float64
 10  Inspection Date  314550 non-null  str    
 11  Inspection Type  314549 non-null  str    
 12  Results          314550 non-null  str    
 13  Violations       225926 non-null  str    
 14  Latitude         313514 non-null  float64
 15  Longitude        313514 non-null  float64
 16  Location         313514 non-null  str    
dtypes:

### Missing Values

In [23]:
summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100).round(2),
})
summary = summary.sort_values("missing_pct", ascending=False)
summary.loc["TOTAL"] = [df.isna().sum().sum(), (df.isna().sum().sum() / df.size * 100).round(2)]
summary.style.format({"missing_count": "{:.0f}", "missing_pct": "{:.2f}%"})

,missing_count,missing_pct
Violations,88629,28.18%
Facility Type,5296,1.68%
AKA Name,2402,0.76%
Longitude,1041,0.33%
Location,1041,0.33%
Latitude,1041,0.33%
City,183,0.06%
State,80,0.03%
Risk,92,0.03%
License #,24,0.01%


### Column Overview

In [24]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "unique": df.nunique(),
    "sample": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
overview

,dtype,non_null,unique,sample
Inspection ID,str,314555,314555,2642587
DBA Name,str,314550,35076,HALAL MANDI HOUSE
AKA Name,str,312153,33370,HALAL MANDI HOUSE
License #,float64,314531,48995,3073458.0
Facility Type,str,309259,527,Restaurant
Risk,str,314463,4,Risk 1 (High)
Address,str,314550,33252,6240 N CALIFORNIA AVE
City,str,314372,94,CHICAGO
State,str,314475,7,IL
Zip,float64,314509,139,60659.0


### Duplicate Analysis

In [25]:
print(f"Fully duplicated rows: {df.duplicated().sum():,}")
print(f"Duplicate Inspection IDs: {df['Inspection ID'].duplicated().sum():,}")
print(f"Unique facilities (License #): {df['License #'].nunique():,}")

Fully duplicated rows: 0
Duplicate Inspection IDs: 0
Unique facilities (License #): 48,995


### Categorical Columns

In [28]:
print(f"\n--- Risk ---")
df["Risk"].value_counts()


--- Risk ---


Risk
Risk 1 (High)      234059
Risk 2 (Medium)     55894
Risk 3 (Low)        24425
All                    85
Name: count, dtype: int64

In [29]:
print(f"\n--- State ---")
df["State"].value_counts()


--- State ---


State
IL    314451
IN        13
CA         5
WI         3
DC         1
CO         1
NY         1
Name: count, dtype: int64

In [31]:
print(f"\n--- Results ---")
df["Results"].value_counts()


--- Results ---


Results
Pass                    162667
Fail                     60512
Pass w/ Conditions       46743
Out of Business          25842
No Entry                 14104
Not Ready                 4587
Business Not Located        95
Name: count, dtype: int64

### Dates

In [37]:
df["Inspection Date"] = pd.to_datetime(df["Inspection Date"], errors="coerce")
print(f"Date range: {df['Inspection Date'].min()} → {df['Inspection Date'].max()}")
print(f"Unparseable dates: {df['Inspection Date'].isna().sum():,}")
print(df["Inspection Date"].dt.year.value_counts().sort_index())

Date range: 2010-01-21 00:00:00 → 2026-09-11 00:00:00
Unparseable dates: 5
Inspection Date
2010.0    17209
2011.0    18747
2012.0    18866
2013.0    20947
2014.0    21540
2015.0    20911
2016.0    22818
2017.0    21587
2018.0    17192
2019.0    19052
2020.0    15123
2021.0    15868
2022.0    16936
2023.0    18261
2024.0    19228
2025.0    19205
2026.0    11060
Name: count, dtype: int64


### Numeric Column Sanity Checks

In [44]:
df.describe(include="number")

,License #,Zip,Latitude,Longitude
count,3.145310e+05,314509.000000,313514.000000,313514.000000
mean,1.806646e+06,60628.694664,41.880597,-87.676388
std,9.480431e+05,193.662236,0.080853,0.058461
min,0.000000e+00,10014.000000,41.644670,-87.906874
25%,1.473123e+06,60614.000000,41.832134,-87.707384
50%,2.103379e+06,60625.000000,41.891733,-87.666350
75%,2.458745e+06,60643.000000,41.939486,-87.634868
max,9.999999e+06,91706.000000,42.021064,-87.525094
